# OD Model Benchmark

Comparison pipeline: **pre-trained GPS models** vs baseline models.

**Workflow:**
1. Train GPS models in `gps_od.ipynb` — weights saved to `results/weights/`
2. Run this notebook to load GPS weights, train baselines, and compare

**Two evaluation modes:**
1. **Single-city** — all models predict on city 48201 (pair-level 80/10/10 split for GPS)
2. **Multi-city (10 cities)** — area-level split: 8 train, 1 val, 1 test


In [ ]:
from dataclasses import replace
import sys, os, gc, time, random, importlib, warnings
from pathlib import Path
from collections import defaultdict
from pprint import pprint

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path(".").resolve()
DATA_PATH = str(PROJECT_ROOT / "data")
sys.path.insert(0, str(PROJECT_ROOT))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")
print(f"Data path: {DATA_PATH}")

## Configurable Model List
Comment/uncomment to select which models to run.

In [ ]:
from models.GPS.config import TrainingConfig, MC_EPOCHS, load_model_config
from models.GPS.config import WEIGHTS_DIR

# ==================== GPS BENCHMARK RUN IDs ====================
# These run_ids must match those trained in gps_od.ipynb SC_CONFIGS / MC_CONFIGS.
# Configs are loaded automatically from saved JSON ({run_id}.json).

GPS_BENCHMARK_IDS = [
    'SC_TF_CE',
    'SC_TF_H',
    'SC_TF_CE_lape',
    'SC_TF_CE_gn',
    'SC_TF_focal',
    'SC_TF_CE_samp',
    'SC_TF_CE_nz',
    'SC_TF_CE_rle',
    'SC_TF_CE_lape_rle',
    'SC_TF_focal_rle',
    'SC_BL_CE',
    'SC_BL_H',
]

GPS_MC_BENCHMARK_IDS = [
    'MC_TF_CE',
    'MC_TF_H',
    'MC_TF_CE_lape',
    'MC_TF_focal',
    'MC_TF_CE_rle',
    'MC_BL_CE',
    'MC_BL_H',
]

# ==================== TRANSFLOWER_ORIG (train inline in benchmark) ====================
# TransFlower paper architecture: MLP encoder + TransFlower decoder + RLE
TRANSFLOWER_ORIG_CONFIG = TrainingConfig(
    encoder_type='mlp', decoder_type='transflower',
    loss_type='ce', prediction_mode='normalized',
    use_dest_sampling=False, use_rle=True,
)

# ==================== BASELINE MODELS ====================
BASELINE_MODELS = [
    'RF',
    'SVR',
    'GBRT',
    'DGM',
    'GM_E',
    'GM_P',
    'GMEL',
    # 'NetGAN',
    # 'DiffODGen',
    # 'WeDAN',
]

SINGLE_CITY_ID = '48201'
MULTI_CITY_IDS = ['17031','48201','04013','06073','06059','36047','12086','48113','06065','36081']

# ==================== BATCH PROCESSING CONFIG ====================
FLAT_BATCH_SIZE = 50_000   # mini-batch size for DGM / GM_E / GM_P DataLoader

trained_sc = [rid for rid in GPS_BENCHMARK_IDS if (WEIGHTS_DIR / f"{rid}.pt").exists()]
trained_mc = [rid for rid in GPS_MC_BENCHMARK_IDS if (WEIGHTS_DIR / f"{rid}.pt").exists()]
print(f"GPS (SC) run_ids: {len(GPS_BENCHMARK_IDS)}  ({len(trained_sc)} trained)")
print(f"GPS (MC) run_ids: {len(GPS_MC_BENCHMARK_IDS)}  ({len(trained_mc)} trained)")
print(f"Baseline models:  {len(BASELINE_MODELS)}")


## Shared Data Utilities

In [ ]:
def load_area_raw(area_id, data_path=DATA_PATH):
    """Load raw npy files for one area."""
    ap = os.path.join(data_path, area_id)
    return {
        'demos': np.load(os.path.join(ap, 'demos.npy')),
        'pois': np.load(os.path.join(ap, 'pois.npy')),
        'adj': np.load(os.path.join(ap, 'adj.npy')),
        'dis': np.load(os.path.join(ap, 'dis.npy')),
        'od': np.load(os.path.join(ap, 'od.npy')),
    }


def get_all_areas(data_path=DATA_PATH):
    """Return sorted list of all area IDs."""
    return sorted([d for d in os.listdir(data_path) if os.path.isdir(os.path.join(data_path, d))])


def split_areas(areas, seed=SEED):
    """80/10/10 area split with fixed seed."""
    areas = list(areas)
    rng = np.random.RandomState(seed)
    rng.shuffle(areas)
    n = len(areas)
    n_train = int(n * 0.8)
    n_val = int(n * 0.9)
    return areas[:n_train], areas[n_train:n_val], areas[n_val:]


def cleanup_gpu():
    """Free GPU memory between model runs."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# Shared standard metrics (copy from models/DGM/metrics.py interface)
from models.GPS.metrics import cal_od_metrics, average_listed_metrics

print(f"Total areas in dataset: {len(get_all_areas())}")

## Model Runners

One runner function per model family. Each returns `list[dict]` of per-area metrics.

In [ ]:
# ==================== FLAT MODEL RUNNER ====================
# For: RF, SVR, GBRT, DGM, GM_E, GM_P
# RF/SVR/GBRT use chunked batch processing to avoid OOM

import math

def _construct_flat_features(areas, data_path=DATA_PATH, feature_mode='full'):
    """Build pairwise feature matrix from areas (float32).
    feature_mode: 'full' = demos+pois+distance (for DGM, RF, SVR, GBRT)
                  'gravity' = pop_total + distance (for GM_E, GM_P)
    Returns: (xs_list, ys_list) — per-area arrays.
    """
    xs, ys = [], []
    for area in areas:
        raw = load_area_raw(area, data_path)
        if feature_mode == 'gravity':
            feat = raw['demos'][:, :1].astype(np.float32)
        else:
            feat = np.concatenate([raw['demos'], raw['pois']], axis=1).astype(np.float32)
        dis = raw['dis'].astype(np.float32)
        n = feat.shape[0]
        feat_o = feat.reshape([n, 1, feat.shape[1]]).repeat(n, axis=1)
        feat_d = feat.reshape([1, n, feat.shape[1]]).repeat(n, axis=0)
        dis_3d = dis.reshape([n, n, 1])
        x = np.concatenate([feat_o, feat_d, dis_3d], axis=2).reshape([-1, feat.shape[1]*2+1])
        y = raw['od'].reshape([-1]).astype(np.float32)
        xs.append(x)
        ys.append(y)
    return xs, ys


def _iter_flat_chunks(areas, data_path, feature_mode, chunk_size=FLAT_CHUNK_SIZE):
    """Yield (x_chunk, y_chunk) float32 for each chunk of areas."""
    for i in range(0, len(areas), chunk_size):
        chunk = areas[i:i+chunk_size]
        xs, ys = _construct_flat_features(chunk, data_path, feature_mode)
        x = np.concatenate(xs, axis=0)
        y = np.concatenate(ys, axis=0)
        del xs, ys
        yield x, y


def _count_chunks(areas, chunk_size=FLAT_CHUNK_SIZE):
    return math.ceil(len(areas) / chunk_size)


def run_flat_model(model_name, train_areas, valid_areas, test_areas, data_path=DATA_PATH):
    """Train and evaluate flat-feature models with batch processing."""
    print(f"\n{'='*60}\n  Running: {model_name}\n{'='*60}")
    t0 = time.time()

    feature_mode = 'gravity' if model_name in ('GM_E', 'GM_P') else 'full'

    # ── RF: warm_start chunked training (all data, batched) ──
    if model_name == 'RF':
        from sklearn.ensemble import RandomForestRegressor

        n_chunks = _count_chunks(train_areas)
        trees_per_chunk = max(2, 20 // n_chunks)
        total_trees = 0

        model = RandomForestRegressor(
            n_estimators=0, warm_start=True,
            max_depth=20, min_samples_split=5, min_samples_leaf=5,
            n_jobs=-1, random_state=SEED
        )
        for ci, (x_chunk, y_chunk) in enumerate(_iter_flat_chunks(train_areas, data_path, feature_mode)):
            total_trees += trees_per_chunk
            model.n_estimators = total_trees
            model.fit(x_chunk, y_chunk)
            print(f"    Chunk {ci+1}/{n_chunks}: {len(x_chunk)} samples, {total_trees} trees")
            del x_chunk, y_chunk; gc.collect()

        print(f"  RF trained: {total_trees} trees total")

    # ── SVR: SGDRegressor with partial_fit (all data, batched) ──
    elif model_name == 'SVR':
        from sklearn.linear_model import SGDRegressor
        from sklearn.preprocessing import StandardScaler

        n_chunks = _count_chunks(train_areas)

        # Pass 1: compute feature scaler
        scaler = StandardScaler()
        total_samples = 0
        for x_chunk, _ in _iter_flat_chunks(train_areas, data_path, feature_mode):
            scaler.partial_fit(x_chunk)
            total_samples += len(x_chunk)
            del x_chunk, _; gc.collect()
        print(f"  Scaler fitted on {total_samples} total samples")

        # Pass 2+: train SGDRegressor (online linear SVR)
        model = SGDRegressor(
            loss='epsilon_insensitive', penalty='l2',
            alpha=1e-4, max_iter=1, tol=None,
            random_state=SEED, warm_start=False
        )
        for epoch in range(FLAT_SGD_EPOCHS):
            for ci, (x_chunk, y_chunk) in enumerate(_iter_flat_chunks(train_areas, data_path, feature_mode)):
                model.partial_fit(scaler.transform(x_chunk), y_chunk)
                del x_chunk, y_chunk; gc.collect()
            print(f"    Epoch {epoch+1}/{FLAT_SGD_EPOCHS} done")

        # Wrap model for unified predict interface
        _sgd_model = model
        _sgd_scaler = scaler
        model = type('SVRWrapper', (), {
            'predict': lambda self, x: _sgd_model.predict(_sgd_scaler.transform(x))
        })()

    # ── GBRT: HistGradientBoostingRegressor (all data, memory-efficient) ──
    elif model_name == 'GBRT':
        from sklearn.ensemble import HistGradientBoostingRegressor

        # Collect all training data in float32 (~275 MB for full dataset)
        x_parts, y_parts = [], []
        for x_chunk, y_chunk in _iter_flat_chunks(train_areas, data_path, feature_mode):
            x_parts.append(x_chunk)
            y_parts.append(y_chunk)
        x_train = np.concatenate(x_parts, axis=0)
        y_train = np.concatenate(y_parts, axis=0)
        del x_parts, y_parts; gc.collect()
        print(f"  GBRT training on {len(x_train)} samples ({x_train.nbytes/1e6:.0f} MB)")

        model = HistGradientBoostingRegressor(
            max_iter=100, max_depth=None,
            min_samples_leaf=2, random_state=SEED
        )
        model.fit(x_train, y_train)
        del x_train, y_train; gc.collect()

    # ── DGM, GM_E, GM_P: PyTorch models (unchanged logic) ──
    elif model_name in ('DGM', 'GM_E', 'GM_P'):
        # Build features (concatenated — these models need all data for torch)
        xs_train, ys_train = _construct_flat_features(train_areas, data_path, feature_mode)
        x_train = np.concatenate(xs_train, axis=0)
        y_train = np.concatenate(ys_train, axis=0)
        del xs_train, ys_train; gc.collect()

        xs_valid, ys_valid = _construct_flat_features(valid_areas, data_path, feature_mode)

        # Dynamic import of model architecture
        model_dir = os.path.join(str(PROJECT_ROOT), 'models', model_name)
        spec = importlib.util.spec_from_file_location('model_mod', os.path.join(model_dir, 'model.py'))
        model_mod = importlib.util.module_from_spec(spec)
        model_mod.np = np
        model_mod.torch = torch
        model_mod.nn = nn
        model_mod.F = F
        spec.loader.exec_module(model_mod)

        if model_name == 'DGM':
            net = model_mod.DeepGravity().cuda()
            feat_scaler = MinMaxScaler((-1, 1)).fit(x_train)
            od_scaler = model_mod.OD_normer(y_train.min(), y_train.max())
            x_tr = torch.FloatTensor(feat_scaler.transform(x_train))
            y_tr = torch.FloatTensor(od_scaler.normalize(y_train))
            ds = torch.utils.data.TensorDataset(x_tr, y_tr)
            del x_tr, y_tr, x_train, y_train; gc.collect()
            dl = torch.utils.data.DataLoader(ds, batch_size=FLAT_BATCH_SIZE, shuffle=True)
            optimizer = torch.optim.Adam(net.parameters(), lr=3e-4)
            best_vl = np.inf; patience = 100
            for ep in range(10000):
                net.train()
                for xb, yb in dl:
                    xb, yb = xb.cuda(), yb.cuda()
                    optimizer.zero_grad()
                    loss = torch.mean((net(xb).squeeze() - yb)**2)
                    loss.backward(); optimizer.step()
                net.eval()
                with torch.no_grad():
                    vls = []
                    for xv, yv in zip(xs_valid, ys_valid):
                        xv_t = torch.FloatTensor(feat_scaler.transform(xv)).cuda()
                        yh = net(xv_t).squeeze().cpu().numpy()
                        vls.append(((yh - od_scaler.normalize(yv))**2).mean())
                    vl = np.mean(vls)
                if vl < best_vl: best_vl = vl; patience = 100
                else:
                    patience -= 1
                    if patience == 0: break
            model = lambda x: od_scaler.renormalize(net(torch.FloatTensor(feat_scaler.transform(x)).cuda()).squeeze().cpu().detach().numpy())

        elif model_name in ('GM_E', 'GM_P'):
            net = model_mod.GRAVITY().cuda()
            ds = torch.utils.data.TensorDataset(
                torch.FloatTensor(x_train), torch.FloatTensor(y_train))
            del x_train, y_train; gc.collect()
            dl = torch.utils.data.DataLoader(ds, batch_size=FLAT_BATCH_SIZE, shuffle=True)
            optimizer = torch.optim.Adam(net.parameters(), lr=1e-4)
            best_vl = np.inf; patience = 100
            for ep in range(10000):
                net.train()
                for xb, yb in dl:
                    xb, yb = xb.cuda(), yb.cuda()
                    optimizer.zero_grad()
                    loss = torch.mean((net(xb).squeeze() - yb) ** 2)
                    loss.backward(); optimizer.step()
                net.eval()
                with torch.no_grad():
                    vls = []
                    for xv, yv in zip(xs_valid, ys_valid):
                        yh = net(torch.FloatTensor(xv).cuda()).squeeze().cpu().numpy()
                        vls.append(((yh - yv)**2).mean())
                    vl = np.mean(vls)
                if vl < best_vl: best_vl = vl; patience = 100
                else:
                    patience -= 1
                    if patience == 0: break
            model = lambda x: net(torch.FloatTensor(x).cuda()).squeeze().cpu().detach().numpy()

    # ── Evaluate on test areas ──
    xs_test, ys_test = _construct_flat_features(test_areas, data_path, feature_mode)
    metrics_all = []
    for x_one, y_one in zip(xs_test, ys_test):
        n = int(np.sqrt(y_one.shape[0]))
        if model_name in ('RF', 'SVR', 'GBRT'):
            y_hat = model.predict(x_one)
        else:  # DGM, GM_E, GM_P
            y_hat = model(x_one)
        y_hat = y_hat.reshape([n, n])
        y_one = y_one.reshape([n, n])
        y_hat[y_hat < 0] = 0
        metrics_all.append(cal_od_metrics(y_hat, y_one))

    avg = average_listed_metrics(metrics_all)
    print(f"  CPC={avg['CPC']:.4f}  RMSE={avg['RMSE']:.2f}  MAE={avg['MAE']:.2f}  ({time.time()-t0:.1f}s)")
    cleanup_gpu()
    return metrics_all

print('Flat model runner defined (with batch processing for RF/SVR/GBRT).')

In [ ]:
# ==================== GRAPH MODEL RUNNER ====================
# For: GMEL, NetGAN

def _load_graph_data(areas, data_path=DATA_PATH):
    """Load per-area data as lists of (nfeat, adj, dis, od)."""
    nfeats, adjs, dises, ods = [], [], [], []
    for area in areas:
        raw = load_area_raw(area, data_path)
        nfeat = np.concatenate([raw['demos'], raw['pois']], axis=1)
        nfeats.append(nfeat)
        adjs.append(raw['adj'])
        dises.append(raw['dis'])
        ods.append(raw['od'])
    return nfeats, adjs, dises, ods


def _get_graph_scalers(nfeats, dises, ods):
    nfeat_scaler = MinMaxScaler().fit(np.concatenate(nfeats, axis=0))
    dis_scaler = MinMaxScaler().fit(np.concatenate([d.reshape(-1,1) for d in dises], axis=0))
    od_scaler = MinMaxScaler().fit(np.concatenate([o.reshape(-1,1) for o in ods], axis=0))
    return nfeat_scaler, dis_scaler, od_scaler


def _iter_graph_areas(areas, data_path=DATA_PATH):
    """Yield (nfeat, adj, dis, od) one area at a time — avoids bulk RAM allocation."""
    for area in areas:
        raw = load_area_raw(area, data_path)
        nfeat = np.concatenate([raw['demos'], raw['pois']], axis=1)
        yield nfeat, raw['adj'], raw['dis'], raw['od']


def run_graph_model(model_name, train_areas, valid_areas, test_areas, data_path=DATA_PATH):
    """Train and evaluate graph-based models (GMEL, NetGAN)."""
    import dgl
    print(f"\n{'='*60}\n  Running: {model_name}\n{'='*60}")
    t0 = time.time()

    # Train areas loaded lazily one at a time; scalers fit on val (small set)
    nf_valid, adj_valid, dis_valid, od_valid = _load_graph_data(valid_areas, data_path)
    nf_test, adj_test, dis_test, od_test = _load_graph_data(test_areas, data_path)
    nfeat_scaler, dis_scaler, od_scaler = _get_graph_scalers(nf_valid, dis_valid, od_valid)

    def build_dgl_graph(adj):
        dst, src = adj.nonzero()
        d = adj[adj.nonzero()]
        g = dgl.graph(([], []))
        g.add_nodes(adj.shape[0])
        g.add_edges(src, dst, {'d': torch.tensor(d).float().view(-1, 1)})
        return g

    # Dynamic import
    model_dir = os.path.join(str(PROJECT_ROOT), 'models', model_name)
    old_path = sys.path.copy()
    sys.path.insert(0, model_dir)

    try:
        if model_name == 'GMEL':
            from sklearn.ensemble import GradientBoostingRegressor
            model_spec = importlib.util.spec_from_file_location('model_mod', os.path.join(model_dir, 'model.py'))
            model_mod = importlib.util.module_from_spec(model_spec)
            model_spec.loader.exec_module(model_mod)

            gmel = model_mod.GMEL().cuda()
            optimizer = torch.optim.Adam(gmel.parameters(), lr=3e-4)
            best_vl = np.inf; patience = 10
            for ep in range(1000):
                gmel.train()
                for nf, adj, dis, od in _iter_graph_areas(train_areas, data_path):
                    nf_s = nfeat_scaler.transform(nf)
                    dis_s = dis_scaler.transform(dis.reshape(-1,1)).reshape(dis.shape)
                    od_s = od_scaler.transform(od.reshape(-1,1)).reshape(od.shape)
                    nf_t = torch.FloatTensor(nf_s).cuda()
                    g = build_dgl_graph(adj).to('cuda')
                    od_t = torch.FloatTensor(od_s).cuda()
                    optimizer.zero_grad()
                    flow_in, flow_out, flow, h_in, h_out = gmel(g, nf_t)
                    loss = (torch.mean((flow_in-od_t.sum(0))**2) +
                            torch.mean((flow_out-od_t.sum(1))**2) +
                            torch.mean((flow-od_t)**2))
                    loss.backward(); optimizer.step()
                # Validation
                gmel.eval()
                with torch.no_grad():
                    vls = []
                    for nf, adj, dis, od in zip(nf_valid, adj_valid, dis_valid, od_valid):
                        nf_s = nfeat_scaler.transform(nf)
                        dis_s = dis_scaler.transform(dis.reshape(-1,1)).reshape(dis.shape)
                        od_s = od_scaler.transform(od.reshape(-1,1)).reshape(od.shape)
                        nf_t = torch.FloatTensor(nf_s).cuda()
                        g = build_dgl_graph(adj).to('cuda')
                        od_t = torch.FloatTensor(od_s).cuda()
                        flow_in, flow_out, flow, _, _ = gmel(g, nf_t)
                        vl = (torch.mean((flow_in-od_t.sum(0))**2) +
                              torch.mean((flow_out-od_t.sum(1))**2) +
                              torch.mean((flow-od_t)**2)).item()
                        vls.append(vl)
                    vl = np.mean(vls)
                if vl < best_vl: best_vl = vl; patience = 10
                else:
                    patience -= 1
                    if patience == 0: break

            # Train GBRT on embeddings (same as GMEL main.py)
            gbrt = GradientBoostingRegressor(n_estimators=20, min_samples_split=2,
                                             min_samples_leaf=2, max_depth=None)
            xtrain_emb, ytrain_emb = [], []
            gmel.eval()
            with torch.no_grad():
                for nf, adj, dis, od in _iter_graph_areas(train_areas, data_path):
                    nf_s = nfeat_scaler.transform(nf)
                    nf_t = torch.FloatTensor(nf_s).cuda()
                    g = build_dgl_graph(adj).to('cuda')
                    _, _, _, h_in, h_out = gmel(g, nf_t)
                    h = np.concatenate([h_in.cpu().numpy(), h_out.cpu().numpy()], axis=1)
                    n = h.shape[0]
                    h_o = h.reshape([n,1,h.shape[1]]).repeat(n,axis=1)
                    h_d = h.reshape([1,n,h.shape[1]]).repeat(n,axis=0)
                    feat = np.concatenate([h_o, h_d, dis.reshape([n,n,1])], axis=2).reshape([-1, h.shape[1]*2+1])
                    xtrain_emb.append(feat)
                    ytrain_emb.append(od.reshape(-1))
            gbrt.fit(np.concatenate(xtrain_emb), np.concatenate(ytrain_emb))

            # Evaluate
            metrics_all = []
            for nf, adj, dis, od in zip(nf_test, adj_test, dis_test, od_test):
                nf_s = nfeat_scaler.transform(nf)
                nf_t = torch.FloatTensor(nf_s).cuda()
                g = build_dgl_graph(adj).to('cuda')
                with torch.no_grad():
                    _, _, _, h_in, h_out = gmel(g, nf_t)
                    h = np.concatenate([h_in.cpu().numpy(), h_out.cpu().numpy()], axis=1)
                    n = h.shape[0]
                    h_o = h.reshape([n,1,h.shape[1]]).repeat(n,axis=1)
                    h_d = h.reshape([1,n,h.shape[1]]).repeat(n,axis=0)
                    feat = np.concatenate([h_o, h_d, dis.reshape([n,n,1])], axis=2).reshape([-1, h.shape[1]*2+1])
                    od_hat = gbrt.predict(feat).reshape([n, n])
                    od_hat[od_hat < 0] = 0
                metrics_all.append(cal_od_metrics(od_hat, od))

        elif model_name == 'NetGAN':
            model_spec = importlib.util.spec_from_file_location('model_mod', os.path.join(model_dir, 'model.py'))
            model_mod = importlib.util.module_from_spec(model_spec)
            model_spec.loader.exec_module(model_mod)

            generator = model_mod.Generator().cuda()
            discriminator = model_mod.Discriminator().cuda()
            opt_G = torch.optim.Adam(generator.parameters(), lr=3e-4)
            opt_D = torch.optim.Adam(discriminator.parameters(), lr=3e-4)

            for epoch in range(2):
                generator.train(); discriminator.train()
                for nf, adj, dis, od in _iter_graph_areas(train_areas, data_path):
                    nf_s = nfeat_scaler.transform(nf)
                    dis_s = dis_scaler.transform(dis.reshape(-1,1)).reshape(dis.shape)
                    nf_t = torch.FloatTensor(nf_s).cuda()
                    g = build_dgl_graph(adj).to('cuda')
                    dis_t = torch.FloatTensor(dis_s).cuda()
                    opt_G.zero_grad()
                    fake_batch = generator.sample_generated_batch(g, nf_t, dis_t, 128).to('cuda')
                    loss_G = -torch.mean(discriminator(fake_batch))
                    loss_G.backward(); opt_G.step()
                    if epoch % 5 == 0:
                        opt_D.zero_grad()
                        with torch.no_grad():
                            _, adjacency, logp = generator.generate_OD_net(g, nf_t, dis_t)
                            batch = [model_mod.sample_one_random_walk(adjacency, logp) for _ in range(128)]
                            fake_batch = torch.stack(batch).to('cuda')
                        od_s = od_scaler.transform(od.reshape(-1,1)).reshape(od.shape)
                        real_batch = torch.FloatTensor(model_mod.sample_batch_real(od_s)).to('cuda')
                        loss_D = (torch.mean(discriminator(fake_batch)) -
                                  torch.mean(discriminator(real_batch)) +
                                  10 * model_mod.compute_gradient_penalty(discriminator, real_batch, fake_batch))
                        loss_D.backward(); opt_D.step()

            # Evaluate
            generator.eval()
            metrics_all = []
            for nf, adj, dis, od in zip(nf_test, adj_test, dis_test, od_test):
                nf_s = nfeat_scaler.transform(nf)
                dis_s = dis_scaler.transform(dis.reshape(-1,1)).reshape(dis.shape)
                nf_t = torch.FloatTensor(nf_s).cuda()
                dis_t = torch.FloatTensor(dis_s).cuda()
                g = build_dgl_graph(adj).to('cuda')
                with torch.no_grad():
                    od_gen, _, _ = generator.generate_OD_net(g, nf_t, dis_t)
                    od_gen = od_gen.cpu().numpy()
                    od_gen = od_scaler.inverse_transform(od_gen)
                    od_gen[od_gen < 0] = 0
                metrics_all.append(cal_od_metrics(od_gen, od))

    finally:
        sys.path = old_path

    avg = average_listed_metrics(metrics_all)
    print(f"  CPC={avg['CPC']:.4f}  RMSE={avg['RMSE']:.2f}  MAE={avg['MAE']:.2f}  ({time.time()-t0:.1f}s)")
    cleanup_gpu()
    return metrics_all

print('Graph model runner defined.')

In [ ]:
# ==================== DIFFUSION MODEL RUNNER ====================
# For: DiffODGen, WeDAN
# These have complex custom pipelines, so we use subprocess execution.

import subprocess, json, tempfile

def run_diffusion_model(model_name, train_areas, valid_areas, test_areas, data_path=DATA_PATH):
    """Run diffusion model via subprocess to avoid import conflicts."""
    print(f"\n{'='*60}\n  Running: {model_name}\n{'='*60}")
    t0 = time.time()

    model_dir = os.path.join(str(PROJECT_ROOT), 'models', model_name)
    old_cwd = os.getcwd()

    try:
        os.chdir(model_dir)
        result = subprocess.run(
            [sys.executable, 'main.py'],
            capture_output=True, text=True, timeout=7200,  # 2h timeout
            cwd=model_dir,
        )
        print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
        if result.returncode != 0:
            print(f"  ERROR: {result.stderr[-300:]}")
            return []
    except subprocess.TimeoutExpired:
        print(f"  TIMEOUT after 2h")
        return []
    finally:
        os.chdir(old_cwd)

    # Parse metrics from stdout (models print pprint(avg_metrics) at the end)
    # This is a best-effort parser
    try:
        lines = result.stdout.strip().split('\n')
        # Find the dict output from pprint
        dict_lines = []
        in_dict = False
        for line in lines:
            if line.strip().startswith('{'):
                in_dict = True
            if in_dict:
                dict_lines.append(line)
            if in_dict and line.strip().endswith('}'):
                break
        if dict_lines:
            dict_str = '\n'.join(dict_lines).replace("'", '"')
            avg_metrics = json.loads(dict_str)
            print(f"  CPC={avg_metrics.get('CPC', 'N/A')}  ({time.time()-t0:.1f}s)")
            return [avg_metrics]  # Single aggregated result
    except Exception as e:
        print(f"  Could not parse metrics: {e}")

    print(f"  Completed in {time.time()-t0:.1f}s (metrics not parsed)")
    return []

print('Diffusion model runner defined.')

In [ ]:
# ==================== GPS RESULTS LOADER ====================
# Loads pre-trained GPS weights (from gps_od.ipynb) and evaluates with full metrics.
# Config is loaded automatically from the saved JSON ({run_id}.json).

from models.GPS.model import make_model
from models.GPS.data_load import prepare_single_city_data, prepare_multi_city_data
from models.GPS.metrics import predict_full_matrix
from models.GPS.config import load_model_config

def load_gps_results(run_id, city_data, config=None):
    """Load pre-trained GPS weights and evaluate. Returns metrics dict or None.
    Config is loaded from saved JSON if available; falls back to the passed config."""
    weight_path = WEIGHTS_DIR / f"{run_id}.pt"
    if not weight_path.exists():
        print(f"  [SKIP] {run_id}: weights not found at {weight_path}")
        print(f"         -> Run gps_od.ipynb first to train this configuration.")
        return None
    # Load saved config if present (ensures model architecture matches weights)
    saved_cfg = load_model_config(run_id)
    effective_cfg = saved_cfg if saved_cfg is not None else config
    if effective_cfg is None:
        print(f"  [SKIP] {run_id}: no saved config JSON and no config passed.")
        return None
    source = "JSON" if saved_cfg is not None else "caller"
    print(f"  Loading {run_id} (config from {source}) ...")
    try:
        model = make_model(effective_cfg, graph_data_ref=city_data['graph_data'])
        model.load_state_dict(torch.load(str(weight_path), map_location=DEVICE))
        model.eval()
        with torch.no_grad():
            pred = predict_full_matrix(model, city_data, effective_cfg)
        pred[pred < 0] = 0
        metrics = cal_od_metrics(pred, city_data['od_matrix_np'])
        print(f"  {run_id}: CPC={metrics['CPC']:.4f}  MAE={metrics['MAE']:.4f}")
        return metrics
    except Exception as e:
        print(f"  ERROR loading {run_id}: {e}")
        return None
    finally:
        cleanup_gpu()

print('GPS results loader defined.')


## Single-City Benchmark

All models predict on city `48201`. OD pairs split 80/10/10 for training.

In [ ]:
print("=" * 70)
print("  SINGLE-CITY BENCHMARK")
print(f"  City: {SINGLE_CITY_ID}")
print("=" * 70)

single_city_results = {}

# ── GPS variants (load pre-trained weights from gps_od.ipynb) ──
print("\n[GPS Models]")
sc_city_data = prepare_single_city_data(area_id=SINGLE_CITY_ID)
for run_id in GPS_BENCHMARK_IDS:
    metrics = load_gps_results(run_id, sc_city_data)
    if metrics:
        single_city_results[run_id] = metrics
cleanup_gpu()

# ── TransFlower orig (train inline) ──
print("\n[TransFlower Orig — train inline]")
from models.GPS.main import train_single_city
try:
    tf_sc_data = prepare_single_city_data(area_id=SINGLE_CITY_ID)
    tf_result = train_single_city(
        'transflower_orig', 'TransFlower Orig (MLP+TF+RLE)',
        TRANSFLOWER_ORIG_CONFIG, city_data=tf_sc_data,
    )
    if tf_result.get('status') == 'ok':
        single_city_results['transflower_orig'] = tf_result['metrics_full']
except Exception as e:
    print(f"  ERROR transflower_orig: {e}")
finally:
    cleanup_gpu()

# ── Baseline models (train inline) ──
print("\n[Baseline Models]")
# Same data as GPS single-city: all baseline models train and eval on SINGLE_CITY_ID
train_areas = [SINGLE_CITY_ID]
valid_areas = [SINGLE_CITY_ID]
test_areas  = [SINGLE_CITY_ID]

for model_name in BASELINE_MODELS:
    try:
        if model_name in ('RF', 'SVR', 'GBRT', 'DGM', 'GM_E', 'GM_P'):
            metrics_list = run_flat_model(model_name, train_areas, valid_areas, test_areas)
        elif model_name in ('GMEL', 'NetGAN'):
            metrics_list = run_graph_model(model_name, train_areas, valid_areas, test_areas)
        elif model_name in ('DiffODGen', 'WeDAN'):
            metrics_list = run_diffusion_model(model_name, train_areas, valid_areas, test_areas)
        else:
            print(f"  Unknown model: {model_name}")
            continue
        if metrics_list:
            single_city_results[model_name] = average_listed_metrics(metrics_list)
    except Exception as e:
        print(f"  ERROR {model_name}: {e}")

print(f"\nCompleted: {len(single_city_results)} models")


In [ ]:
# ── Single-city results table ──
if single_city_results:
    df_single = pd.DataFrame(single_city_results).T
    # Select key columns if they exist
    key_cols = ['CPC', 'CPC_nonzero', 'RMSE', 'MAE', 'MAPE', 'SMAPE',
                'accuracy', 'matrix_COS_similarity', 'JSD_inflow', 'JSD_outflow', 'JSD_ODflow']
    available_cols = [c for c in key_cols if c in df_single.columns]
    if available_cols:
        df_single = df_single[available_cols]
    df_single = df_single.sort_values('CPC', ascending=False)

    print("\n" + "=" * 100)
    print("  SINGLE-CITY RESULTS (sorted by CPC)")
    print("=" * 100)
    print(df_single.to_string())

    # Save
    os.makedirs('results', exist_ok=True)
    df_single.to_csv('results/single_city_benchmark.csv')
    print(f"\nSaved to results/single_city_benchmark.csv")
else:
    print("No results to display.")

## Multi-City (10 Cities) Benchmark

10 cities, area-level split: 8 train, 1 val, 1 test.

In [ ]:
print("=" * 70)
print("  MULTI-CITY BENCHMARK")
print(f"  Cities: {MULTI_CITY_IDS}")
print("=" * 70)

np.random.seed(SEED)
mc_ids = MULTI_CITY_IDS.copy()
rng_mc = np.random.RandomState(SEED)
rng_mc.shuffle(mc_ids)
mc_train = mc_ids[:8]
mc_valid = mc_ids[8:9]
mc_test  = mc_ids[9:]
print(f"Train: {mc_train}")
print(f"Valid: {mc_valid}")
print(f"Test:  {mc_test}")

multi_city_results = {}

# ── GPS variants (load pre-trained weights) ──
print("\n[GPS Models]")
from models.GPS.data_load import prepare_multi_city_data as _mc_load
from models.GPS.config import load_model_config as _load_cfg
_mc_data_cache = {}

def _get_mc_city_data(pe_type='rwpe'):
    if pe_type not in _mc_data_cache:
        cd, _, _, _ = _mc_load(pe_type=pe_type)
        _mc_data_cache[pe_type] = cd
    return _mc_data_cache[pe_type]

for run_id in GPS_MC_BENCHMARK_IDS:
    saved_cfg = _load_cfg(run_id)
    pe = saved_cfg.pe_type if saved_cfg else 'rwpe'
    mc_city_data = _get_mc_city_data(pe)
    per_city_metrics = []
    for cid, cd in mc_city_data.items():
        m = load_gps_results(run_id, cd)
        if m:
            per_city_metrics.append(m)
    if per_city_metrics:
        multi_city_results[run_id] = average_listed_metrics(per_city_metrics)
cleanup_gpu()

# ── TransFlower orig (train inline) ──
print("\n[TransFlower Orig — train inline]")
from models.GPS.main import train_multi_city
try:
    tf_mc_cfg = replace(TRANSFLOWER_ORIG_CONFIG, mc_epochs=MC_EPOCHS)
    tf_mc_dict, tf_train_ids, tf_val_ids, _ = _mc_load(pe_type=tf_mc_cfg.pe_type)
    tf_mc_result = train_multi_city(
        'MC_transflower_orig', 'MC TransFlower Orig (MLP+TF+RLE)',
        tf_mc_cfg,
        city_data_dict=tf_mc_dict,
        train_city_ids=tf_train_ids,
        val_city_ids=tf_val_ids,
    )
    if tf_mc_result.get('status') == 'ok':
        multi_city_results['MC_transflower_orig'] = tf_mc_result['metrics_full']
except Exception as e:
    print(f"  ERROR MC_transflower_orig: {e}")
finally:
    cleanup_gpu()

# ── Baseline models ──
print("\n[Baseline Models]")
for model_name in BASELINE_MODELS:
    try:
        if model_name in ('RF', 'SVR', 'GBRT', 'DGM', 'GM_E', 'GM_P'):
            metrics_list = run_flat_model(model_name, mc_train, mc_valid, mc_test)
        elif model_name in ('GMEL', 'NetGAN'):
            metrics_list = run_graph_model(model_name, mc_train, mc_valid, mc_test)
        elif model_name in ('DiffODGen', 'WeDAN'):
            metrics_list = run_diffusion_model(model_name, mc_train, mc_valid, mc_test)
        else:
            continue
        if metrics_list:
            multi_city_results[model_name] = average_listed_metrics(metrics_list)
    except Exception as e:
        print(f"  ERROR {model_name}: {e}")

print(f"\nCompleted: {len(multi_city_results)} models")


In [ ]:
# ── Multi-city results table ──
if multi_city_results:
    df_multi = pd.DataFrame(multi_city_results).T
    key_cols = ['CPC', 'CPC_nonzero', 'RMSE', 'MAE', 'MAPE', 'SMAPE',
                'accuracy', 'matrix_COS_similarity', 'JSD_inflow', 'JSD_outflow', 'JSD_ODflow']
    available_cols = [c for c in key_cols if c in df_multi.columns]
    if available_cols:
        df_multi = df_multi[available_cols]
    df_multi = df_multi.sort_values('CPC', ascending=False)

    print("\n" + "=" * 100)
    print("  MULTI-CITY RESULTS (sorted by CPC)")
    print("=" * 100)
    print(df_multi.to_string())

    df_multi.to_csv('results/multi_city_benchmark.csv')
    print(f"\nSaved to results/multi_city_benchmark.csv")
else:
    print("No results to display.")

## Visualization

In [ ]:
def plot_comparison(results_dict, title, metrics_to_plot=None):
    """Bar chart comparison of models."""
    if not results_dict:
        print("No results to plot.")
        return
    if metrics_to_plot is None:
        metrics_to_plot = ['CPC', 'MAE', 'RMSE']

    df = pd.DataFrame(results_dict).T
    available = [m for m in metrics_to_plot if m in df.columns]
    if not available:
        print("No matching metrics found.")
        return

    n_metrics = len(available)
    fig, axes = plt.subplots(1, n_metrics, figsize=(6 * n_metrics, 6))
    if n_metrics == 1:
        axes = [axes]

    # Color by model family
    colors = []
    for name in df.index:
        if 'GPS' in name:
            colors.append('#2196F3')  # blue for GPS
        elif name in ('DiffODGen', 'WeDAN'):
            colors.append('#9C27B0')  # purple for diffusion
        elif name in ('GMEL', 'NetGAN'):
            colors.append('#FF9800')  # orange for graph
        else:
            colors.append('#4CAF50')  # green for classical

    for i, metric in enumerate(available):
        vals = df[metric].values
        axes[i].barh(range(len(df)), vals, color=colors)
        axes[i].set_yticks(range(len(df)))
        axes[i].set_yticklabels(df.index, fontsize=9)
        axes[i].set_xlabel(metric)
        axes[i].set_title(metric)
        axes[i].grid(axis='x', alpha=0.3)
        # Highlight best
        if metric in ('CPC', 'accuracy', 'matrix_COS_similarity'):
            best_idx = np.argmax(vals)
        else:
            best_idx = np.argmin(vals)
        axes[i].barh(best_idx, vals[best_idx], color='red', alpha=0.7)

    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


# Plot single-city results
plot_comparison(single_city_results, f'Single-City Benchmark ({SINGLE_CITY_ID})',
                ['CPC', 'MAE', 'RMSE'])

# Plot multi-city results
plot_comparison(multi_city_results, 'Multi-City Benchmark (10 cities)',
                ['CPC', 'MAE', 'RMSE'])

In [ ]:
# ── Combined summary ──
print("\n" + "=" * 120)
print("  COMBINED SUMMARY")
print("=" * 120)

all_models = set(list(single_city_results.keys()) + list(multi_city_results.keys()))
summary_rows = []
for model in sorted(all_models):
    row = {'Model': model}
    if model in single_city_results:
        sc = single_city_results[model]
        row['SC_CPC'] = sc.get('CPC', None)
        row['SC_MAE'] = sc.get('MAE', None)
        row['SC_RMSE'] = sc.get('RMSE', None)
    if model in multi_city_results:
        mc = multi_city_results[model]
        row['MC_CPC'] = mc.get('CPC', None)
        row['MC_MAE'] = mc.get('MAE', None)
        row['MC_RMSE'] = mc.get('RMSE', None)
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows).set_index('Model')
print(df_summary.to_string())

df_summary.to_csv('results/benchmark_combined.csv')
print(f"\nSaved to results/benchmark_combined.csv")